In [7]:
import h3
import geopandas as gpd
import polars as pl

In [35]:
RESOLUTION = 9

In [36]:
# load df with turbines
turbine_gdf = gpd.read_file("../../../ml/data/geoJSON/wind_us_farms.geojson")

In [37]:
turbine_gdf = turbine_gdf.to_crs(epsg=4326)
centroids = turbine_gdf.geometry.to_crs(3857).centroid.to_crs(4326)
turbine_gdf['longitude'] = centroids.x
turbine_gdf['latitude'] = centroids.y

In [38]:
turbine_gdf = turbine_gdf[['fid', 'longitude', 'latitude']]
turbine_gdf.shape

(78090, 3)

In [39]:
res = 9

turbine_cells = (
    pl.from_records(
        [
            (h3.latlng_to_cell(lat, lon, res),)
            for lat, lon in zip(
                turbine_gdf["latitude"],
                turbine_gdf["longitude"],
            )
        ],
        schema=["h3_index"],
        orient="row"
    )
    .unique()
)


In [40]:
turbine_cells.shape

(66162, 1)

In [41]:
df = pl.DataFrame(turbine_cells, schema=["h3_index"])
df = df.with_columns(
    pl.lit(1).alias("has_turbine")
)

df.head()

h3_index,has_turbine
str,i32
"""8926db4b167ffff""",1
"""8926662294fffff""",1
"""89260c12963ffff""",1
"""892631a0667ffff""",1
"""8926d3524afffff""",1


In [42]:
# Use a US states GeoJSON — or load your own
us_geojson = gpd.read_file("../data/US_StateBoundaries.geojson")
us_geojson["NAME"].unique()

<ArrowStringArray>
[            'Arkansas',              'Arizona',             'Colorado',
                 'Iowa',             'Illinois',              'Indiana',
               'Kansas',             'Missouri',             'Nebraska',
           'New Mexico',               'Nevada',             'Oklahoma',
            'Tennessee',                 'Utah',        'West Virginia',
                'Idaho',              'Montana',         'North Dakota',
         'South Dakota',              'Vermont',              'Wyoming',
          'Puerto Rico',  'U.S. Virgin Islands',              'Florida',
                'Texas',              'Alabama',              'Georgia',
            'Louisiana',          'Mississippi',       'South Carolina',
           'California',          'Connecticut', 'District of Columbia',
             'Delaware',             'Kentucky',        'Massachusetts',
             'Maryland',       'North Carolina',           'New Jersey',
             'New York',        

In [43]:
us_geojson = us_geojson[~us_geojson["NAME"].isin(["U.S. Virgin Islands", "Hawaii", "Puerto Rico", "Alaska"])]

In [44]:
us_boundary = us_geojson.geometry.union_all()

In [58]:
n = 200_000

points = gpd.GeoSeries([us_boundary], crs="EPSG:4326").sample_points(size=n).iloc[0]
cells = [h3.latlng_to_cell(p.y, p.x, RESOLUTION) for p in points.geoms]


In [59]:
turbine_cells = df["h3_index"].to_list()
cells = cells + turbine_cells

In [61]:
final_cells = list(set(cells))
len(final_cells)


265705

In [63]:
turbine_cells

['8926db4b167ffff',
 '8926662294fffff',
 '89260c12963ffff',
 '892631a0667ffff',
 '8926d3524afffff',
 '8926d869e8fffff',
 '89489b2d237ffff',
 '8926c3b9833ffff',
 '8926da10e47ffff',
 '8926e1364b7ffff',
 '892993a5063ffff',
 '892a8562d83ffff',
 '89488b79997ffff',
 '89488b78b03ffff',
 '89263548277ffff',
 '8926d311863ffff',
 '892a333a547ffff',
 '8926e676907ffff',
 '894889c9367ffff',
 '8926d305c93ffff',
 '8926c6ab6afffff',
 '8926cc54383ffff',
 '8926a87684bffff',
 '8928892b6abffff',
 '8926f148bcfffff',
 '89288d16853ffff',
 '8948919b6afffff',
 '8926663122bffff',
 '8926c6485dbffff',
 '8926dab5253ffff',
 '8926f508b9bffff',
 '8926e16b5c7ffff',
 '8926c0041c7ffff',
 '8926db8d1cfffff',
 '8926d18a58bffff',
 '8912c90db6bffff',
 '8926d0619d7ffff',
 '8926da0ad5bffff',
 '8926760d1abffff',
 '89283052d27ffff',
 '8928b26f113ffff',
 '8926db1658fffff',
 '8926db77303ffff',
 '892662baad7ffff',
 '8926d869e03ffff',
 '89267236073ffff',
 '892685a11afffff',
 '8926d2223dbffff',
 '89279ab22dbffff',
 '8926d3b982bffff',


In [64]:
final_df = pl.DataFrame(final_cells, schema=["h3_index"])

turbine_set = set(turbine_cells)

final_df = pl.DataFrame(final_cells, schema=["h3_index"]).with_columns(
    pl.col("h3_index").is_in(turbine_set).alias("has_turbine")
)


In [67]:
# Add lat and lon
final_df = final_df.with_columns(
    pl.col("h3_index")
    .map_elements(
        lambda x: list(h3.cell_to_latlng(x)),
        return_dtype=pl.List(pl.Float64),
    )
    .alias("_ll")
).with_columns(
    pl.col("_ll").list.get(0).alias("lat"),
    pl.col("_ll").list.get(1).alias("lon"),
).drop("_ll")


In [69]:
print(final_df.head())
print(final_df.shape)

shape: (5, 4)
┌─────────────────┬─────────────┬───────────┬─────────────┐
│ h3_index        ┆ has_turbine ┆ lat       ┆ lon         │
│ ---             ┆ ---         ┆ ---       ┆ ---         │
│ str             ┆ bool        ┆ f64       ┆ f64         │
╞═════════════════╪═════════════╪═══════════╪═════════════╡
│ 892742091bbffff ┆ false       ┆ 45.139191 ┆ -88.871743  │
│ 8926844f62bffff ┆ false       ┆ 41.490223 ┆ -107.566305 │
│ 892999ced4fffff ┆ false       ┆ 39.973303 ┆ -115.772628 │
│ 8926d3b82bbffff ┆ true        ┆ 34.749517 ┆ -102.265162 │
│ 8928d5cd31bffff ┆ false       ┆ 47.461155 ┆ -122.344325 │
└─────────────────┴─────────────┴───────────┴─────────────┘
(265705, 4)


In [70]:
final_df.write_csv("../data/h3_cells.csv")